# kloppy

## Import libraries

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from kloppy import statsbomb, wyscout

In [2]:
# Add the project root to the Python path
sys.path.append(str(Path.cwd().parents[0]))

from config import project_paths

In [3]:
# Ignore specific warnings for cleaner output
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

## Load providers data

In [4]:
# StatsBomb match and player IDs for testing (UEFA Euro 2024 Final - Lamine Yamal)
MATCH_ID = 3943043
PLAYER_ID = 316046
PROVIDER = "statsbomb"

In [5]:
# Load dataset based on provider
if PROVIDER == "statsbomb":
    dataset = statsbomb.load(
        event_data=project_paths.STATSBOMB_EVENTS_DIR / f"{MATCH_ID}.json",
        lineup_data=project_paths.STATSBOMB_LINEUPS_DIR / f"{MATCH_ID}.json",
    )
elif PROVIDER == "wyscout":
    dataset = wyscout.load(
        event_data=project_paths.WYSCOUT_PROCESSED_V2_DIR / f"{MATCH_ID}.json",
    )
else:
    raise ValueError("Unsupported provider. Use 'statsbomb' or 'wyscout'.")

## Metadata

### Teams

In [6]:
# Print home and away team names
home_team, away_team = dataset.metadata.teams
print(f"Home team: {home_team.name}")
print(f"Away team: {away_team.name}")

Home team: Spain
Away team: England


### Time

In [7]:
# Print all periods
for period in dataset.metadata.periods:
    print(f"{period}\n")

Period(id=1, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=2822, microseconds=704000), prev_period=None, next_period=Period(id=2, start_timestamp=datetime.timedelta(seconds=2822, microseconds=704000), end_timestamp=datetime.timedelta(seconds=5764, microseconds=393000), prev_period=..., next_period=None))

Period(id=2, start_timestamp=datetime.timedelta(seconds=2822, microseconds=704000), end_timestamp=datetime.timedelta(seconds=5764, microseconds=393000), prev_period=Period(id=1, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=2822, microseconds=704000), prev_period=None, next_period=...), next_period=None)



In [8]:
# Print start and end time for each period
for period in dataset.metadata.periods:
    print(f"Start time: {period.start_time}")
    print(f"End time: {period.end_time}")

Start time: P1T00:00
End time: P1T47:03
Start time: P2T00:00
End time: P2T49:02


In [9]:
# Get match start and end time
match_start_time = dataset.metadata.periods[0].start_time
match_end_time = dataset.metadata.periods[-1].end_time

print(f"Match start time: {match_start_time}")
print(f"Match end time: {match_end_time}")

Match start time: P1T00:00
Match end time: P2T49:02


### Players

In [10]:
# Print all players for each team
for team in dataset.metadata.teams:
    print(f"Players for team {team.name}:")
    print(team.players)
    print()

Players for team Spain:
[Player(player_id='3042', team=Team(team_id='772', name='Spain', ground=home, starting_formation=<FormationType.FOUR_TWO_THREE_ONE: '4-2-3-1'>, formations=<TimeContainer>, players=[...]), jersey_no=6, first_name=None, last_name=None, name='Mikel Merino Zazón', starting=False, starting_position=None, positions=TimeContainer[PositionType]({'P2T43:41': <PositionType.RightWing: ('Right Wing', 'RW', 'WideMidfield')>, 'P2T44:54': <PositionType.CenterAttackingMidfield: ('Center Attacking Midfield', 'CAM', 'AttackingMidfield')>}), attributes={}), Player(player_id='3265', team=Team(team_id='772', name='Spain', ground=home, starting_formation=<FormationType.FOUR_TWO_THREE_ONE: '4-2-3-1'>, formations=<TimeContainer>, players=[...]), jersey_no=9, first_name=None, last_name=None, name='José Luis Sanmartín Mato', starting=False, starting_position=None, positions=<TimeContainer>, attributes={}), Player(player_id='3477', team=Team(team_id='772', name='Spain', ground=home, start

In [11]:
# Show player information and positions for each team
for team in dataset.metadata.teams:
    print(f"Team: {team.name}")
    for player in team.players:
        print(f"  Player: {player.name}")
        print(f"    Jersey Number: {player.jersey_no}")
        for start_time, end_time, position in player.positions.ranges():
            if position is not None:
                print(f"    Position: {position.code} from {start_time} to {end_time}")
    print(f"Total players in {team.name}: {len(team.players)}\n")

Team: Spain
  Player: Mikel Merino Zazón
    Jersey Number: 6
    Position: RW from P2T43:41 to P2T44:54
    Position: CAM from P2T44:54 to P2T49:02
  Player: José Luis Sanmartín Mato
    Jersey Number: 9
  Player: Álvaro Borja Morata Martín
    Jersey Number: 7
    Position: ST from P1T00:00 to P2T22:19
  Player: David Raya Martin
    Jersey Number: 1
  Player: Aymeric Laporte
    Jersey Number: 14
    Position: LCB from P1T00:00 to P2T49:02
  Player: José Ignacio Fernández Iglesias
    Jersey Number: 4
    Position: RCB from P2T37:38 to P2T49:02
  Player: Daniel Carvajal Ramos
    Jersey Number: 2
    Position: RB from P1T00:00 to P2T49:02
  Player: Fabián Ruiz Peña
    Jersey Number: 8
    Position: LDM from P1T00:00 to P2T49:02
  Player: Mikel Oyarzabal Ugarte
    Jersey Number: 21
    Position: ST from P2T22:19 to P2T49:02
  Player: Ferrán Torres García
    Jersey Number: 11
  Player: Rodrigo Hernández Cascante
    Jersey Number: 16
    Position: RDM from P1T00:00 to P2T00:00
  Pl

### Substitutions

In [12]:
# Display player positions and indicate substitutions (in/out) for each team
for team in dataset.metadata.teams:
    print(f"Team: {team.name}")

    for player in team.players:
        print(f"  Player: {player.name} (#{player.jersey_no})")

        # Get player positions and their intervals
        positions = list(player.positions.ranges())

        # Skip players without position data
        if not positions:
            print("    No position data available.")
            continue

        # Show all positions and their intervals
        for start_time, end_time, position in positions:
            if position is not None:
                print(f"    Position: {position.code} from {start_time} to {end_time}")

        # Indicate if the player entered as a substitute
        if not player.starting:
            print(f"    → Entered as substitute at: {positions[0][0]}")

        # Indicate if the player was substituted out (left before match end)
        last_end_time = positions[-1][1]
        if last_end_time != match_end_time:
            print(f"    → Substituted out at: {last_end_time}")
    print()

Team: Spain
  Player: Mikel Merino Zazón (#6)
    Position: RW from P2T43:41 to P2T44:54
    Position: CAM from P2T44:54 to P2T49:02
    → Entered as substitute at: P2T43:41
  Player: José Luis Sanmartín Mato (#9)
    No position data available.
  Player: Álvaro Borja Morata Martín (#7)
    Position: ST from P1T00:00 to P2T22:19
    → Substituted out at: P2T22:19
  Player: David Raya Martin (#1)
    No position data available.
  Player: Aymeric Laporte (#14)
    Position: LCB from P1T00:00 to P2T49:02
  Player: José Ignacio Fernández Iglesias (#4)
    Position: RCB from P2T37:38 to P2T49:02
    → Entered as substitute at: P2T37:38
  Player: Daniel Carvajal Ramos (#2)
    Position: RB from P1T00:00 to P2T49:02
  Player: Fabián Ruiz Peña (#8)
    Position: LDM from P1T00:00 to P2T49:02
  Player: Mikel Oyarzabal Ugarte (#21)
    Position: ST from P2T22:19 to P2T49:02
    → Entered as substitute at: P2T22:19
  Player: Ferrán Torres García (#11)
    No position data available.
  Player: Rod

## Data exploration

### Find all

In [13]:
# Find all goals in the dataset
dataset.find_all("shot.goal")

[StatsBombShotEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.LEFT_FOOT: 'LEFT_FOOT'>)]),
 StatsBombShotEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.LEFT_FOOT: 'LEFT_FOOT'>)]),
 StatsBombShotEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.RIGHT_FOOT: 'RIGHT_FOOT'>)])]

### DataFrame

In [14]:
# Convert dataset to DataFrame for easier exploration
df = dataset.to_df()

In [15]:
# Display the first 5 rows of the dataset
df.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,end_coordinates_y,receiver_player_id,set_piece_type,body_part_type,pass_type,is_under_pressure,duel_type,is_counter_attack,goalkeeper_type,card_type
0,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
1,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
2,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
3,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
4,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,1,0 days 00:00:00.340000,0 days 00:00:02.869454,alive,768,768,99174,0.499564,...,0.48318,3468,KICK_OFF,RIGHT_FOOT,None,None,None,None,None,None


In [16]:
# Display dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   event_id            3415 non-null   object         
 1   event_type          3415 non-null   object         
 2   period_id           3415 non-null   int64          
 3   timestamp           3415 non-null   timedelta64[ns]
 4   end_timestamp       1676 non-null   timedelta64[ns]
 5   ball_state          3415 non-null   object         
 6   ball_owning_team    3415 non-null   object         
 7   team_id             3415 non-null   object         
 8   player_id           3400 non-null   object         
 9   coordinates_x       3389 non-null   float64        
 10  coordinates_y       3389 non-null   float64        
 11  result              1844 non-null   object         
 12  success             1844 non-null   object         
 13  end_coordinates_x   1701 non-null

## Filter columns from the dataset

In [17]:
# Filter columns from the dataset
filtered_df = dataset.to_df(
    "player_id",
    "player",
    "team_id",
    "team",
    "event_id",
    "event_type",
    "result",
    "success",
    "body_part_type",
    "pass_type",
    "duel_type",
    "set_piece_type",
    "goalkeeper_type",
    "card_type",
    "coordinates_x",
    "coordinates_y",
    "time",
)

In [18]:
# Create a mapping for dtypes of all columns
dtype_mapping = {
    "player_id": "Int64",
    "player": "string",
    "team_id": "Int64",
    "team": "string",
    "event_id": "string",
    "event_type": "category",
    "result": "category",
    "success": "boolean",
    "body_part_type": "category",
    "pass_type": "category",
    "duel_type": "category",
    "set_piece_type": "category",
    "goalkeeper_type": "category",
    "card_type": "category",
    "coordinates_x": "Float64",
    "coordinates_y": "Float64",
    "time": "string",
}

# Convert the type of all DataFrame columns
filtered_df = filtered_df.astype(dtype_mapping)

# Verify changes
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3415 non-null   Int64   
 3   team             3415 non-null   string  
 4   event_id         3415 non-null   string  
 5   event_type       3415 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3389 non-null   Float64 
 15  coordinates_y    3389 non-null   Float64 
 16  time             3415 non-null   string  


In [19]:
# Display DataFrame
filtered_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
0,<NA>,<NA>,772,Spain,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
1,<NA>,<NA>,768,England,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
2,<NA>,<NA>,768,England,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
3,<NA>,<NA>,772,Spain,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00


## Group events by player

In [20]:
# Filter DataFrame to only keeps rows related to player events
players_df = filtered_df[filtered_df["player_id"].notna()]
players_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00
5,3468,Jordan Pickford,768,England,d64668c7-747c-4a7d-912c-e1c3ff357a67,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
6,3468,Jordan Pickford,768,England,9c107df3-a3c8-4ad5-bc35-00214087a105,CARRY,COMPLETE,True,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
7,3468,Jordan Pickford,768,England,237201b8-aef8-4823-b282-e82875795c07,PASS,OUT,False,LEFT_FOOT,LONG_BALL,NaN,NaN,NaN,NaN,0.244381,0.386189,P1T00:05
8,22084,Bukayo Saka,768,England,c979e198-edc1-4f22-851a-26cedb6474cf,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.798231,0.721654,P1T00:10


In [21]:
# Count events by player
event_count_by_player_df = players_df.groupby(["player", "event_type"]).size().unstack()

# Remove columns axis name and convert "player" from index to a column
event_count_by_player_df = event_count_by_player_df.rename_axis(None, axis="columns").reset_index()

# Display the event count by player
event_count_by_player_df.head()

,player,BALL_OUT,CARD,CARRY,CLEARANCE,DUEL,FORMATION_CHANGE,FOUL_COMMITTED,GENERIC:Ball Receipt*,GENERIC:Block,...,GENERIC:Starting XI,GOALKEEPER,INTERCEPTION,MISCONTROL,PASS,PRESSURE,RECOVERY,SHOT,SUBSTITUTION,TAKE_ON
0,Aymeric Laporte,2,0,71,7,7,0,0,74,2,...,0,0,0,0,83,2,1,1,0,0
1,Bukayo Saka,4,0,29,1,3,0,2,32,2,...,0,0,0,0,24,17,3,0,0,1
2,Cole Palmer,1,0,8,1,1,0,0,7,0,...,0,0,0,0,7,7,2,1,0,1
3,Daniel Carvajal Ramos,4,0,50,2,8,0,1,54,3,...,0,0,2,0,73,15,5,0,0,2
4,Daniel Olmo Carvajal,1,1,32,0,3,0,1,40,2,...,0,0,1,2,32,26,3,2,0,1


In [22]:
# Display DataFrame info
event_count_by_player_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   player                     29 non-null     string
 1   BALL_OUT                   29 non-null     int64 
 2   CARD                       29 non-null     int64 
 3   CARRY                      29 non-null     int64 
 4   CLEARANCE                  29 non-null     int64 
 5   DUEL                       29 non-null     int64 
 6   FORMATION_CHANGE           29 non-null     int64 
 7   FOUL_COMMITTED             29 non-null     int64 
 8   GENERIC:Ball Receipt*      29 non-null     int64 
 9   GENERIC:Block              29 non-null     int64 
 10  GENERIC:Dispossessed       29 non-null     int64 
 11  GENERIC:Dribbled Past      29 non-null     int64 
 12  GENERIC:Error              29 non-null     int64 
 13  GENERIC:Foul Won           29 non-null     int64 
 14  GENERIC:Goal

## Combine Events with Result and Success

In [23]:
# Create comprehensive player statistics with success ratios
def create_player_statistics(df):
    """Create detailed player statistics including success ratios and sub-types"""

    # Start with basic event counts
    basic_events = df.groupby(["player", "event_type"]).size().unstack(fill_value=0)

    # Events that have success ratios - we'll exclude these from basic events
    events_with_results = ["pass", "shot", "take_on", "carry", "interception", "duel"]

    # Remove events that have ratios from basic_events to avoid duplication
    basic_events = basic_events.drop(columns=[col for col in events_with_results if col in basic_events.columns])

    # Create success ratio format for events with results
    success_ratios = {}

    for event in events_with_results:
        if event in df["event_type"].values:
            event_data = df[df["event_type"] == event]

            # Count total and successful for each player
            total_counts = event_data.groupby("player").size()
            success_counts = event_data[event_data["success"] == True].groupby("player").size()

            # Create ratio format "successful/total"
            ratio_series = total_counts.index.map(
                lambda player: f"{success_counts.get(player, 0)}/{total_counts.get(player, 0)}"
            )

            success_ratios[f"{event}_ratio"] = pd.Series(ratio_series.values, index=total_counts.index)

    # Add sub-type details (body_part, pass_type, etc.)
    subtype_columns = ["body_part_type", "pass_type", "duel_type", "set_piece_type", "goalkeeper_type", "card_type"]

    subtype_stats = {}
    for col in subtype_columns:
        if col in df.columns:
            # Convert to lowercase before grouping
            df_copy = df.copy()
            df_copy[col] = df_copy[col].astype(str).str.lower()

            # Count occurrences of each subtype per player
            subtype_counts = (
                df_copy[df_copy[col].notna() & (df_copy[col] != "nan")]
                .groupby(["player", col])
                .size()
                .unstack(fill_value=0)
            )

            # Rename columns to include the type prefix and convert to int
            subtype_counts.columns = [f"{col}_{subcol}" for subcol in subtype_counts.columns]
            subtype_counts = subtype_counts.astype(int)
            subtype_stats[col] = subtype_counts

    return basic_events, success_ratios, subtype_stats


# Apply the function
basic_events, success_ratios, subtype_stats = create_player_statistics(players_df)

print("Basic events shape:", basic_events.shape)
print("Success ratios created:", list(success_ratios.keys()))
print("Subtype categories:", list(subtype_stats.keys()))

Basic events shape: (29, 29)
Success ratios created: []
Subtype categories: ['body_part_type', 'pass_type', 'duel_type', 'set_piece_type', 'goalkeeper_type', 'card_type']


In [24]:
# Create DataFrame with player-team mapping
player_team_mapping = players_df[["player", "team"]].drop_duplicates().reset_index(drop=True)
player_team_mapping.head()

,player,team
0,Kobbie Mainoo,England
1,Jordan Pickford,England
2,Bukayo Saka,England
3,Unai Simón Mendibil,Spain
4,Robin Aime Robert Le Normand,Spain


In [25]:
# Merge all statistics into final comprehensive table
def merge_comprehensive_stats(basic_events, success_ratios, subtype_stats, team_mapping):
    """Merge basic events, success ratios, and subtypes into one comprehensive DataFrame"""

    # Start with basic events and reset index to make player a column
    final_df = basic_events.reset_index()

    # Add team information right after player
    final_df = final_df.merge(team_mapping, on="player", how="left")

    # Reorder to have player, team, then basic events
    basic_cols = [col for col in final_df.columns if col not in ["player", "team"]]
    final_df = final_df[["player", "team"] + basic_cols]

    # Add success ratios
    for ratio_name, ratio_series in success_ratios.items():
        ratio_df = ratio_series.reset_index()
        ratio_df.columns = ["player", ratio_name]
        final_df = final_df.merge(ratio_df, on="player", how="left")

    # Add subtype statistics
    for subtype_name, subtype_df in subtype_stats.items():
        if not subtype_df.empty:
            subtype_reset = subtype_df.reset_index()
            final_df = final_df.merge(subtype_reset, on="player", how="left")

    # Fill NaN values - use 0 for numeric columns and "0/0" for ratios
    ratio_columns = [col for col in final_df.columns if col.endswith("_ratio")]

    # Fill numeric columns with 0 and convert to int where appropriate
    numeric_cols = final_df.select_dtypes(include=["float64", "int64"]).columns
    final_df[numeric_cols] = final_df[numeric_cols].fillna(0).astype(int)

    # Fill ratio columns with "0/0"
    for col in ratio_columns:
        final_df[col] = final_df[col].fillna("0/0")

    return final_df


# Create the comprehensive statistics table
comprehensive_stats = merge_comprehensive_stats(basic_events, success_ratios, subtype_stats, player_team_mapping)

print("Comprehensive player statistics:")
print(f"Shape: {comprehensive_stats.shape}")
print(f"Columns: {list(comprehensive_stats.columns)}")
print("\nFirst 3 players:")
comprehensive_stats.head()

Comprehensive player statistics:
Shape: (29, 57)
Columns: ['player', 'team', 'BALL_OUT', 'CARD', 'CARRY', 'CLEARANCE', 'DUEL', 'FORMATION_CHANGE', 'FOUL_COMMITTED', 'GENERIC:Ball Receipt*', 'GENERIC:Block', 'GENERIC:Dispossessed', 'GENERIC:Dribbled Past', 'GENERIC:Error', 'GENERIC:Foul Won', 'GENERIC:Goal Keeper', 'GENERIC:Half End', 'GENERIC:Half Start', 'GENERIC:Injury Stoppage', 'GENERIC:Referee Ball-Drop', 'GENERIC:Shield', 'GENERIC:Starting XI', 'GOALKEEPER', 'INTERCEPTION', 'MISCONTROL', 'PASS', 'PRESSURE', 'RECOVERY', 'SHOT', 'SUBSTITUTION', 'TAKE_ON', 'body_part_type_both_hands', 'body_part_type_head', 'body_part_type_keeper_arm', 'body_part_type_left_foot', 'body_part_type_no_touch', 'body_part_type_other', 'body_part_type_right_foot', 'body_part_type_right_hand', 'pass_type_cross', 'pass_type_hand_pass', 'pass_type_head_pass', 'pass_type_high_pass', 'pass_type_long_ball', 'pass_type_shot_assist', 'duel_type_aerial', 'duel_type_ground', 'duel_type_loose_ball', 'set_piece_type_

,player,team,BALL_OUT,CARD,CARRY,CLEARANCE,DUEL,FORMATION_CHANGE,FOUL_COMMITTED,GENERIC:Ball Receipt*,...,duel_type_loose_ball,set_piece_type_corner_kick,set_piece_type_free_kick,set_piece_type_goal_kick,set_piece_type_kick_off,set_piece_type_throw_in,goalkeeper_type_claim,goalkeeper_type_punch,goalkeeper_type_save,card_type_first_yellow
0,Aymeric Laporte,Spain,2,0,71,7,7,0,0,74,...,0,0,1,2,0,0,0,0,0,0
1,Bukayo Saka,England,4,0,29,1,3,0,2,32,...,0,0,0,0,0,0,0,0,0,0
2,Cole Palmer,England,1,0,8,1,1,0,0,7,...,0,1,1,0,0,0,0,0,0,0
3,Daniel Carvajal Ramos,Spain,4,0,50,2,8,0,1,54,...,0,0,0,0,0,14,0,0,0,0
4,Daniel Olmo Carvajal,Spain,1,1,32,0,3,0,1,40,...,0,0,0,0,0,0,0,0,0,2
